# Week 7 Exercise - Fine-tuning Open-Source LLMs for Product Price Prediction

This notebook extends Week 6 by fine-tuning open-source Hugging Face models for price prediction using:
- LoRA (normal fine-tune)
- QLoRA 8-bit
- QLoRA 4-bit

It also includes a Gradio app to compare predictions across variants, plus evaluation and error analysis.

## Colab Setup

> Recommended: run this notebook on Google Colab with GPU (T4 or better).

In [ ]:
# Uncomment in Colab if needed
# !pip -q install -U transformers datasets peft trl accelerate bitsandbytes gradio scikit-learn pandas numpy python-dotenv

In [ ]:
!pip install -q --upgrade python-decouple


In [ ]:
import os
import re
import json
import time
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import gradio as gr

from datasets import load_dataset, Dataset
from huggingface_hub import list_repo_files, login
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

from decouple import config

# Optional: token can come from Colab userdata or environment

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# -----------------------------
# Core configuration
# -----------------------------
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Colab-friendly base model
HF_CATEGORY = "raw_meta_Appliances"
MAX_ROWS = 15000

TEST_SIZE = 0.15
VAL_SIZE_FROM_TRAIN = 0.1

MAX_TEXT_CHARS = 1400
MIN_TEXT_CHARS = 40

MAX_TRAIN_SAMPLES = 3000
MAX_VAL_SAMPLES = 400
MAX_TEST_SAMPLES = 300

# Training toggles
RUN_LORA_FULL = False
RUN_QLORA_8BIT = False
RUN_QLORA_4BIT = False

OUTPUT_DIR = Path("./week7_price_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANT_DIRS = {
    "lora_full": OUTPUT_DIR / "lora_full",
    "qlora_8bit": OUTPUT_DIR / "qlora_8bit",
    "qlora_4bit": OUTPUT_DIR / "qlora_4bit",
}

for p in VARIANT_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

PROMPT_TEMPLATES = {
    "short": "Estimate product price in USD. Return JSON only: {\"price_usd\": number}.",
    "standard": "You are a pricing assistant. Estimate the product price in USD from the description. Return strict JSON only: {\"price_usd\": number}.",
    "detailed": "You are an expert retail pricing assistant. Infer a realistic USD price from title/features/description/details. Return strict JSON only: {\"price_usd\": number}. No extra text.",
}


def resolve_hf_token() -> Optional[str]:
    """
    Resolve Hugging Face token ONLY from Google Colab userdata.
    The userdata key name is sourced from config("HR_TOKEN").
    Example: if config("HF_TOKEN") returns "HF_TOKEN", we call userdata.get("HF_TOKEN").
    """
    secret_key_name = config("HF_TOKEN", default=None)
    if not secret_key_name:
        print("config('HF_TOKEN') is missing. Set HF_TOKEN to your Colab userdata key name.")
        return None

    try:
        from google.colab import userdata  # type: ignore
        token_value = userdata.get(secret_key_name)
        if token_value:
            print(f"Using token from Colab userdata key: {secret_key_name}")
            return token_value
        print(f"No value found in Colab userdata for key: {secret_key_name}")
        return None
    except Exception as e:
        print(f"Could not read Colab userdata: {e}")
        return None


HF_TOKEN = config("HF_TOKEN")
if HF_TOKEN:
    try:
        login(HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login successful")
    except Exception as e:
        print(f"Hugging Face login warning: {e}")
else:
    print("HF token unavailable from Colab userdata; using anonymous access")

print("Configured base model:", BASE_MODEL)

## Data preparation (Week 6 style)

In [ ]:
def build_text(row: Dict) -> str:
    title = (row.get("title") or "").strip()
    features = " ".join(row.get("features") or [])
    description = " ".join(row.get("description") or [])
    details = row.get("details")
    if isinstance(details, dict):
        details_text = " ".join(f"{k}: {v}" for k, v in details.items())
    else:
        details_text = str(details or "")

    text = f"{title}. {features} {description} {details_text}".strip()
    text = re.sub(r"\s+", " ", text)
    return text


def clean_row(row: Dict) -> Optional[Dict]:
    try:
        price = float(row.get("price"))
    except (TypeError, ValueError):
        return None

    if not (1 <= price <= 1000):
        return None

    text = build_text(row)
    if len(text) < MIN_TEXT_CHARS:
        return None

    return {
        "text": text[:MAX_TEXT_CHARS],
        "price": float(price),
    }


def _load_raw_meta_direct(category: str):
    """Try known raw meta file patterns directly (no script loader)."""
    repo = "McAuley-Lab/Amazon-Reviews-2023"
    c = category.replace("raw_meta_", "")

    candidates = [
        f"hf://datasets/{repo}/raw/meta_categories/meta_{c}.parquet",
        f"hf://datasets/{repo}/raw/meta_categories/meta_{c}.jsonl.gz",
        f"hf://datasets/{repo}/raw/meta_categories/meta_{c}.json.gz",
        f"hf://datasets/{repo}/raw/meta_categories/meta_{c}.json",
        f"hf://datasets/{repo}/raw/meta_categories/{c}.parquet",
        f"hf://datasets/{repo}/raw/meta_categories/{c}.jsonl.gz",
        f"hf://datasets/{repo}/raw/meta_categories/{c}.json.gz",
        f"hf://datasets/{repo}/raw/meta_categories/{c}.json",
    ]

    errors = []
    for path in candidates:
        try:
            if path.endswith(".parquet"):
                return load_dataset("parquet", data_files=[path], split="train", token=HF_TOKEN)
            return load_dataset("json", data_files=[path], split="train", token=HF_TOKEN)
        except Exception as e:
            errors.append(str(e))

    return None


def _load_week6_fallback():
    """Fallback to script-free curated dataset used in Week 6."""
    ds = load_dataset("ed-donner/items_lite", split="train", token=HF_TOKEN)
    rows = []
    for x in ds:
        try:
            p = float(x.get("price"))
        except Exception:
            continue
        s = (x.get("summary") or "").strip()
        if 1 <= p <= 1000 and len(s) >= MIN_TEXT_CHARS:
            rows.append({"text": s[:MAX_TEXT_CHARS], "price": p})
    return rows


def load_clean_items(category: str = HF_CATEGORY, max_rows: int = MAX_ROWS) -> List[Dict]:
    raw_ds = _load_raw_meta_direct(category)

    if raw_ds is not None:
        if max_rows and len(raw_ds) > max_rows:
            raw_ds = raw_ds.select(range(max_rows))
        cleaned = [clean_row(x) for x in raw_ds]
        cleaned = [c for c in cleaned if c is not None]
        if cleaned:
            return cleaned

    print("Raw meta category not available in this environment. Falling back to ed-donner/items_lite.")
    fallback_rows = _load_week6_fallback()
    if max_rows and len(fallback_rows) > max_rows:
        fallback_rows = fallback_rows[:max_rows]
    if not fallback_rows:
        raise ValueError("Could not load data from raw meta files or fallback dataset.")
    return fallback_rows

In [ ]:
items = load_clean_items(HF_CATEGORY, MAX_ROWS)
print(f"Clean items: {len(items):,}")

train_items, test_items = train_test_split(items, test_size=TEST_SIZE, random_state=SEED)
train_items, val_items = train_test_split(train_items, test_size=VAL_SIZE_FROM_TRAIN, random_state=SEED)

train_items = train_items[:MAX_TRAIN_SAMPLES]
val_items = val_items[:MAX_VAL_SAMPLES]
test_items = test_items[:MAX_TEST_SAMPLES]

print(f"Train: {len(train_items):,}, Val: {len(val_items):,}, Test: {len(test_items):,}")
print("Example:", train_items[0])

In [ ]:
def build_messages(description: str, template_name: str = "standard") -> List[Dict[str, str]]:
    system_prompt = PROMPT_TEMPLATES[template_name]
    user_prompt = (
        "Product description:\n"
        f"{description}\n\n"
        "Return strict JSON only."
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def training_text(item: Dict, tokenizer: AutoTokenizer, template_name: str = "standard") -> str:
    messages = build_messages(item["text"], template_name=template_name)
    answer = json.dumps({"price_usd": round(float(item["price"]), 2)})
    messages.append({"role": "assistant", "content": answer})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def inference_text(description: str, tokenizer: AutoTokenizer, template_name: str = "standard") -> str:
    messages = build_messages(description, template_name=template_name)
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## Tokenizer and tokenized datasets

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_sft_dataset(items_subset: List[Dict], template_name: str = "standard") -> Dataset:
    texts = [training_text(x, tokenizer, template_name=template_name) for x in items_subset]
    return Dataset.from_dict({"text": texts})

train_ds = make_sft_dataset(train_items)
val_ds = make_sft_dataset(val_items)

MAX_LENGTH = 512

def tokenize_batch(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    return tokenized

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(train_tok[0].keys())

In [ ]:
@dataclass
class VariantConfig:
    name: str
    quantization: Optional[str] = None  # None, "8bit", "4bit"
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05


def get_quant_config(quantization: Optional[str]):
    if quantization == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    if quantization == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    return None


def build_model_for_variant(cfg: VariantConfig):
    quant_cfg = get_quant_config(cfg.quantization)

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        token=HF_TOKEN,
        quantization_config=quant_cfg,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )

    if cfg.quantization in ("8bit", "4bit"):
        model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


def train_variant(cfg: VariantConfig, output_dir: Path, epochs: int = 1, batch_size: int = 2, grad_accum: int = 8):
    model = build_model_for_variant(cfg)

    args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=2e-4,
        num_train_epochs=epochs,
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=80,
        save_steps=120,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        fp16=False,
        report_to="none",
        gradient_checkpointing=True,
        dataloader_num_workers=2,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=collator,
    )

    trainer.train()
    trainer.model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print(f"Saved adapter: {output_dir}")

In [ ]:
# Optional training runs (toggle from config cell)

if RUN_LORA_FULL:
    train_variant(
        VariantConfig(name="lora_full", quantization=None),
        VARIANT_DIRS["lora_full"],
        epochs=1,
        batch_size=2,
        grad_accum=8,
    )

if RUN_QLORA_8BIT:
    train_variant(
        VariantConfig(name="qlora_8bit", quantization="8bit"),
        VARIANT_DIRS["qlora_8bit"],
        epochs=1,
        batch_size=2,
        grad_accum=8,
    )

if RUN_QLORA_4BIT:
    train_variant(
        VariantConfig(name="qlora_4bit", quantization="4bit"),
        VARIANT_DIRS["qlora_4bit"],
        epochs=1,
        batch_size=2,
        grad_accum=8,
    )

## Evaluation utilities (validator, retry, latency, error analysis)

In [ ]:
LOADED_MODELS: Dict[str, Tuple[AutoModelForCausalLM, AutoTokenizer]] = {}


def adapter_exists(path: Path) -> bool:
    return (path / "adapter_config.json").exists()


def parse_price_from_output(text: str) -> Tuple[Optional[float], bool]:
    if not text:
        return None, False

    # Try strict JSON first
    try:
        maybe_json = json.loads(text.strip())
        if isinstance(maybe_json, dict) and "price_usd" in maybe_json:
            val = float(maybe_json["price_usd"])
            return max(0.0, val), True
    except Exception:
        pass

    # Fallback regex
    match = re.search(r"\d+(?:\.\d+)?", text.replace(",", ""))
    if not match:
        return None, False
    try:
        return max(0.0, float(match.group(0))), False
    except ValueError:
        return None, False


def load_variant_model(variant: str):
    if variant in LOADED_MODELS:
        return LOADED_MODELS[variant]

    adapter_dir = VARIANT_DIRS.get(variant)
    if adapter_dir is None or not adapter_exists(adapter_dir):
        raise ValueError(f"Adapter not found for variant '{variant}'. Train it first.")

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        token=HF_TOKEN,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    tok = AutoTokenizer.from_pretrained(str(adapter_dir), trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model.eval()

    LOADED_MODELS[variant] = (model, tok)
    return model, tok


def generate_price(
    model,
    tok,
    description: str,
    template_name: str = "standard",
    max_new_tokens: int = 40,
    temperature: float = 0.2,
):
    prompt = inference_text(description, tok, template_name=template_name)
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            pad_token_id=tok.eos_token_id,
        )

    text = tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text


def predict_with_retry(
    variant: str,
    description: str,
    template_name: str = "standard",
    retries: int = 2,
):
    model, tok = load_variant_model(variant)

    last_text = ""
    for _ in range(retries + 1):
        last_text = generate_price(model, tok, description, template_name=template_name)
        value, strict_json = parse_price_from_output(last_text)
        if value is not None:
            return value, strict_json, last_text

    return None, False, last_text

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate_variant(variant: str, test_subset: List[Dict], template_name: str = "standard"):
    records = []
    y_true, y_pred = [], []
    strict_count = 0
    parse_success = 0
    latencies = []

    for row in test_subset:
        start = time.perf_counter()
        pred, strict_json, raw = predict_with_retry(variant, row["text"], template_name=template_name)
        elapsed = time.perf_counter() - start

        latencies.append(elapsed)
        actual = float(row["price"])

        if pred is not None:
            parse_success += 1
            strict_count += int(strict_json)
            y_true.append(actual)
            y_pred.append(float(pred))

        records.append({
            "variant": variant,
            "actual": actual,
            "pred": pred,
            "abs_error": abs(actual - pred) if pred is not None else None,
            "strict_json": strict_json,
            "latency_s": elapsed,
            "raw_output": raw,
            "text": row["text"][:250],
        })

    df = pd.DataFrame(records)

    metrics = {
        "variant": variant,
        "samples": len(test_subset),
        "parse_success_rate": parse_success / max(1, len(test_subset)),
        "strict_json_rate": strict_count / max(1, len(test_subset)),
        "avg_latency_s": float(np.mean(latencies)) if latencies else None,
        "mae": float(mean_absolute_error(y_true, y_pred)) if y_true else None,
        "rmse": rmse(y_true, y_pred) if y_true else None,
    }
    return metrics, df


def error_analysis(eval_df: pd.DataFrame, top_n: int = 10):
    clean = eval_df.dropna(subset=["abs_error"]).copy()
    if clean.empty:
        return clean, clean, pd.DataFrame()

    over = clean.sort_values("abs_error", ascending=False).head(top_n)
    under = clean.sort_values("abs_error", ascending=False).tail(top_n)

    bins = [0, 25, 75, 150, 300, 1000]
    labels = ["0-25", "25-75", "75-150", "150-300", "300+"]
    clean["price_bucket"] = pd.cut(clean["actual"], bins=bins, labels=labels, include_lowest=True)
    bucket_stats = clean.groupby("price_bucket", as_index=False)["abs_error"].mean().rename(columns={"abs_error": "bucket_mae"})

    return over, under, bucket_stats

In [ ]:
# Optional: run offline evaluations after training variants

VARIANTS_TO_EVAL = [
    v for v in ["lora_full", "qlora_8bit", "qlora_4bit"] if adapter_exists(VARIANT_DIRS[v])
]

metrics_rows = []
eval_outputs = {}

for v in VARIANTS_TO_EVAL:
    m, df = evaluate_variant(v, test_items, template_name="standard")
    metrics_rows.append(m)
    eval_outputs[v] = df

leaderboard_df = pd.DataFrame(metrics_rows)
if not leaderboard_df.empty:
    leaderboard_df = leaderboard_df.sort_values("mae", na_position="last")

leaderboard_df

## Gradio app (single, compare, error analysis, export)

In [ ]:
EXPORT_DIR = OUTPUT_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def available_variants() -> List[str]:
    return [v for v in ["lora_full", "qlora_8bit", "qlora_4bit"] if adapter_exists(VARIANT_DIRS[v])]


def model_card_markdown() -> str:
    rows = []
    rows.append("### Model Card Summary")
    rows.append(f"- Base model: `{BASE_MODEL}`")
    rows.append(f"- Dataset category: `{HF_CATEGORY}`")
    rows.append(f"- Train/Val/Test sizes: {len(train_items)} / {len(val_items)} / {len(test_items)}")
    rows.append("- Variants expected: `lora_full`, `qlora_8bit`, `qlora_4bit`")
    if 'leaderboard_df' in globals() and isinstance(leaderboard_df, pd.DataFrame) and not leaderboard_df.empty:
        best = leaderboard_df.iloc[0]
        rows.append(f"- Best MAE so far: `{best['variant']}` -> {best['mae']:.2f}")
    return "\n".join(rows)


def single_predict(description: str, variant: str, template_name: str, retries: int):
    if not description or not description.strip():
        return "Please enter a description.", "", ""

    if variant not in available_variants():
        return f"Variant `{variant}` not available. Train or load adapter first.", "", ""

    start = time.perf_counter()
    pred, strict_json, raw = predict_with_retry(
        variant=variant,
        description=description.strip(),
        template_name=template_name,
        retries=int(retries),
    )
    latency = time.perf_counter() - start

    if pred is None:
        summary = f"No valid price parsed. Latency: {latency:.2f}s"
    else:
        summary = f"Predicted price: ${pred:.2f} | Strict JSON: {strict_json} | Latency: {latency:.2f}s"

    return summary, raw, model_card_markdown()


def compare_predict(description: str, template_name: str, retries: int):
    variants = available_variants()
    if not variants:
        return pd.DataFrame([{"message": "No trained adapters found."}])

    rows = []
    for v in variants:
        start = time.perf_counter()
        pred, strict_json, raw = predict_with_retry(v, description, template_name=template_name, retries=int(retries))
        latency = time.perf_counter() - start
        rows.append({
            "variant": v,
            "pred_price": pred,
            "strict_json": strict_json,
            "latency_s": round(latency, 3),
            "raw_output": raw,
        })
    return pd.DataFrame(rows)


def analysis_for_variant(variant: str, top_n: int):
    if 'eval_outputs' not in globals() or variant not in eval_outputs:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), None

    over, under, buckets = error_analysis(eval_outputs[variant], top_n=top_n)

    export_path = EXPORT_DIR / f"{variant}_eval.csv"
    eval_outputs[variant].to_csv(export_path, index=False)
    return over, under, buckets, str(export_path)

In [ ]:
with gr.Blocks(title="Week 7 - Open-Source LLM Fine-Tuning Comparator") as demo:
    gr.Markdown("## Week 7 Product Price Predictor - LoRA / QLoRA Comparator")
    gr.Markdown("Compare fine-tuned variants: full LoRA, 8-bit QLoRA, 4-bit QLoRA")

    with gr.Tab("Single Prediction"):
        desc_in = gr.Textbox(label="Product Description", lines=8)
        variant_in = gr.Dropdown(
            choices=["lora_full", "qlora_8bit", "qlora_4bit"],
            value="qlora_4bit",
            label="Variant",
        )
        template_in = gr.Dropdown(choices=list(PROMPT_TEMPLATES.keys()), value="standard", label="Prompt Template")
        retries_in = gr.Slider(0, 3, value=2, step=1, label="Validator retries")
        single_btn = gr.Button("Predict")

        single_summary = gr.Textbox(label="Summary")
        single_raw = gr.Textbox(label="Raw model output", lines=5)
        model_card_out = gr.Markdown(value=model_card_markdown())

        single_btn.click(
            fn=single_predict,
            inputs=[desc_in, variant_in, template_in, retries_in],
            outputs=[single_summary, single_raw, model_card_out],
        )

    with gr.Tab("Compare Variants"):
        compare_desc = gr.Textbox(label="Product Description", lines=8)
        compare_template = gr.Dropdown(choices=list(PROMPT_TEMPLATES.keys()), value="standard", label="Prompt Template")
        compare_retries = gr.Slider(0, 3, value=2, step=1, label="Validator retries")
        compare_btn = gr.Button("Compare All Available Variants")
        compare_table = gr.Dataframe(label="Comparison")

        compare_btn.click(
            fn=compare_predict,
            inputs=[compare_desc, compare_template, compare_retries],
            outputs=[compare_table],
        )

    with gr.Tab("Error Analysis + Export"):
        analysis_variant = gr.Dropdown(
            choices=["lora_full", "qlora_8bit", "qlora_4bit"],
            value="qlora_4bit",
            label="Variant",
        )
        top_n_in = gr.Slider(5, 30, value=10, step=1, label="Top N")
        analysis_btn = gr.Button("Run Analysis")

        over_df = gr.Dataframe(label="Largest errors")
        under_df = gr.Dataframe(label="Smallest errors")
        bucket_df = gr.Dataframe(label="Bucket MAE")
        export_file = gr.File(label="Download evaluation CSV")

        analysis_btn.click(
            fn=analysis_for_variant,
            inputs=[analysis_variant, top_n_in],
            outputs=[over_df, under_df, bucket_df, export_file],
        )

# In Colab environments, you may need share=True
# demo.launch(share=True)
demo.launch()